In [9]:
from mlx_lm import load, generate
import re
import json
import ast

In [ ]:
SYSTEM_PROMPT = """You are a writer specializing in travel reviews in Spanish or English.
Classify ONLY the tourist review text as SUGGESTION or NON-SUGGESTION.
IMPORTANT: If the text includes a management response (e.g., "Dear traveler...", "Thank you for your visit..."), ignore only these sections, focus on the remaining review text.

## DEFINITIONS
1. **SUGGESTION**:
    - Must be an **EXPLICIT** request for change, addition, or removal of a feature of the product/service offered.
    - The recipient is the **MANAGEMENT/ADMINISTRATION**.
    - It is a concrete, actionable request to improve the TOURIST EXPERIENCE.
2. **NON-SUGGESTION**:
    - **Descriptive complaints**: "The bed was hard" or "Service was slow" (Past facts, not requests for change).
    - **Compliments**: "Everything was excellent."
    - **Advice to third parties (CRITICAL)**: "Go early," "Ask for table 4," "I don't recommend it for families." This is NOT a suggestion to management; it is advice for other tourists.
3. SPECULATIVE INFERENCES vs. SUGGESTIONS: Phrases expressing high probability or assumptions about the place's beauty (e.g., "It must be beautiful at night," "It must be spectacular in the evening") are INFERENCES, not suggestions. These are compliments based on potential, not requests for management action. A suggestion requires a "gap" between the current state and a desired future state.

## DECISION TREE

STEP 1 — WHO IS THE TOURIST TALKING TO?
Read the fragment and ask: is the tourist addressing other visitors,
or expressing something toward the service itself?
- Describing the place or experience for other readers → NON-SUGGESTION.
- Describing personal feelings or reactions without addressing management → NON-SUGGESTION.
- Advising other tourists what to do, bring, or expect → NON-SUGGESTION.
- Expressing how the service could or should be different → continue to Step 2.
- Feelings, experiences, or suggestions to other tourists about other attractions or products → NON-SUGGESTION.
- ⚠️ "You" in English reviews is usually impersonal (= "one" / "a visitor"),
  not a direct address to management. "You would enjoy it more if..." or
  "Perhaps you could..." are tourist-facing observations, not management requests.

STEP 2 — IS THERE A DESIRED FUTURE STATE?
Does the tourist express a gap between how the service IS and how they WANT it to be?
- Only describes what happened (past facts, complaints) → NON-SUGGESTION.
- Expresses praise, satisfaction, or intent to return → NON-SUGGESTION.
- Expresses a wish, desire or expectation for something different → continue to Step 3.
- ⚠️ Speculative or hypothetical framing ("perhaps", "maybe", "might", "could",
  "quizás", "tal vez") signals an observation, not a request. A genuine desire
  for change uses direct framing: "they should", "I wish they had", "it would
  help if", "falta", "sería bueno que", "deberían".
- Expresses a concrete wish or expectation for something different → continue to Step 3.

STEP 3 — CAN MANAGEMENT ACT ON IT?
Could the service's management implement a concrete change based on this?
- Vague feeling or emotional reaction without actionable content → NON-SUGGESTION.
- Concrete change in offering, communication, pricing policy, or operations → SUGGESTION.

## RESPONSE
Reply ONLY with this JSON, no additional text:
{
  "razonamiento_corto": "Step 1: [tourists/management]. Step 2: [future gap/no]. Step 3: [actionable/no]. [1 sentence conclusion]",
  "etiqueta": "SUGGESTION" or "NON-SUGGESTION"
}"""

In [11]:
def _clean_model_response(text: str) -> str:
    cleaned = str(text)
    cleaned = re.sub(r"<think>.*?</think>", "", cleaned, flags=re.DOTALL | re.IGNORECASE)
    cleaned = cleaned.replace("</think>", "")
    cleaned = cleaned.replace("<think>", "")
    cleaned = re.sub(r"```(?:json)?", "", cleaned, flags=re.IGNORECASE)
    cleaned = cleaned.replace("```", "")
    return cleaned.strip()

def _extract_braced_blocks(text: str) -> list[str]:
    blocks: list[str] = []
    depth = 0
    start_idx = None

    for idx, char in enumerate(text):
        if char == "{":
            if depth == 0:
                start_idx = idx
            depth += 1
        elif char == "}" and depth > 0:
            depth -= 1
            if depth == 0 and start_idx is not None:
                blocks.append(text[start_idx : idx + 1])
                start_idx = None

    return blocks

def _coerce_to_dict(candidate: str) -> dict | None:
    # 1) JSON estricto
    try:
        parsed = json.loads(candidate)
        if isinstance(parsed, dict):
            return parsed
    except json.JSONDecodeError:
        pass

    # 2) Dict estilo Python (comillas simples, etc.)
    try:
        parsed = ast.literal_eval(candidate)
        if isinstance(parsed, dict):
            return parsed
    except (ValueError, SyntaxError):
        pass

    return None

def _normalize_label(raw_label: str | None) -> str | None:
    if not raw_label:
        return None
    normalized = str(raw_label).strip().upper()
    # Normaliza variantes de guion frecuentes en respuestas LLM.
    normalized = re.sub(r"[\u2010\u2011\u2012\u2013\u2014\u2212_]", "-", normalized)
    normalized = re.sub(r"\s+", "", normalized)
    if normalized == "NON-SUGGESTION":
        return "NON-SUGGESTION"
    if normalized == "SUGGESTION":
        return "SUGGESTION"
    return None

def _extract_fields_without_json(cleaned_text: str) -> dict | None:
    label_match = re.search(
        r'["\']?etiqueta["\']?\s*:\s*["\']?(SUGGESTION|NON[-_\u2010\u2011\u2012\u2013\u2014\u2212]SUGGESTION)["\']?',
        cleaned_text,
        flags=re.IGNORECASE,
    )
    reason_match = re.search(
        r'["\']?razonamiento_corto["\']?\s*:\s*["\'](.+?)["\']\s*(?:,|\}|$)',
        cleaned_text,
        flags=re.IGNORECASE | re.DOTALL,
    )

    label = _normalize_label(label_match.group(1) if label_match else None)
    reason = reason_match.group(1).strip() if reason_match else None

    if not label:
        # Fallback: buscar veredicto textual al final de la respuesta.
        verdict_match = re.search(
            r"(?:therefore|final(?:\s+label)?|clas(?:sification|ificacion)|etiqueta)\W+"
            r"(SUGGESTION|NON[-_\u2010\u2011\u2012\u2013\u2014\u2212]SUGGESTION)",
            cleaned_text,
            flags=re.IGNORECASE,
        )
        label = _normalize_label(verdict_match.group(1) if verdict_match else None)

    if label:
        if not reason:
            sentence = cleaned_text.split("\n")[0].strip()
            reason = sentence[:280] if sentence else "Etiqueta inferida desde respuesta parcial"
        return {
            "razonamiento_corto": reason,
            "etiqueta": label,
        }

    return None

def extract_json_from_olmo(text: str) -> dict | None:
    try:
        cleaned = _clean_model_response(text)

        # Intento 1: extraer todos los bloques {...} balanceados y evaluar desde el final.
        # En este caso, normalmente el ultimo bloque contiene la respuesta final.
        for candidate in reversed(_extract_braced_blocks(cleaned)):
            parsed = _coerce_to_dict(candidate)
            if not isinstance(parsed, dict):
                continue

            label = _normalize_label(parsed.get("etiqueta"))
            reason = str(parsed.get("razonamiento_corto", "")).strip()
            if label:
                return {
                    "razonamiento_corto": reason,
                    "etiqueta": label,
                }

        # Intento 2: JSON truncado al final por limite de tokens.
        first_open = cleaned.rfind("{")
        if first_open != -1:
            candidate = cleaned[first_open:]
            open_count = candidate.count("{")
            close_count = candidate.count("}")
            if open_count > close_count:
                candidate += "}" * (open_count - close_count)

            parsed = _coerce_to_dict(candidate)
            if isinstance(parsed, dict):
                label = _normalize_label(parsed.get("etiqueta"))
                reason = str(parsed.get("razonamiento_corto", "")).strip()
                if label:
                    return {
                        "razonamiento_corto": reason,
                        "etiqueta": label,
                    }

        # Intento 3: extraer campos aunque el JSON no sea parseable.
        salvaged = _extract_fields_without_json(cleaned)
        if salvaged: return salvaged
        return None

    except Exception as e:
        return None

def analyze_data(data, model, tokenizer):
    message = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": f"""Analiza la siguiente reseña:
            ---
            {data}
            ---
            Respuesta en JSON:"""
        }
    ]

    prompt = tokenizer.apply_chat_template(
        message,
        tokenize=False,
        add_generation_prompt=True
    )

    response = generate(
        model,
        tokenizer,
        prompt= prompt,
        max_tokens=2500,
        verbose=False,
    )
    extracted_json = extract_json_from_olmo(response)

    return extracted_json

In [12]:
model_id = 'mlx-community/olmo-3-7b-think-4bit' #"mlx-community/olmo-3-7b-think-4bit"
model, tokenizer = load(model_id)

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

`rope_parameters`'s beta_fast field must be a float, got 32
`rope_parameters`'s beta_slow field must be a float, got 1


In [14]:
problem_list = [
    """There is a bit of confusion in buying tickets - you need to be prepared to wait if there aren't enough people to make the trip. Also, you need to understand Spanish as we did not find any offer of English translation. The boat trip had lovely scenery, and the big ships in port, and the sea lions were just as interesting to the kids as the buoy. Was also a nice idea to have a (cheap) photo waiting for us when we returned to the pier.""",
    """An amazing place... As soon as you arrive in San Pedro, you start to notice the mountain range where the Moon Valley is located, leaving you with the feeling that you have to go right away. It’s a very hot place, but bearable because it’s very windy—though the sun is very strong. The great thing is that the Moon Valley tour has different stops like the 'Tres Marías,' the old salt mine extraction area, caves formed by water, and finally watching the sunset. Everything was incredible! As I mentioned before, you don't need to dress warmly for this tour since it's hot; even at sunset, there’s just a strong wind, but it isn't cold.""",
    """We booked our tours with ED, we closed 06/07/08 August 2024 , were super honest, wonderful tours, punctual , we returned to the hotel tired of so much to walk, can close tour of vcs with tranquility , galera is good people , driver and guide super friendly with the brazilian galera ! Note: one of the tours we won a picnic , very top congratulations Alfatur""",
    """We hiked to the suspension bridges on our adventure hiking the W trek. Just too far away to REALLY appreciate the size and magnitude. Perhaps on one of the boat trips you would be able to appreciate it more.""",
    """The place is overwhelming because of its sheer size. The surrounding vegetation is beautiful. Perhaps the walking paths and each station inside the cave could be better marked. There is also a circuit that leads above the cave, which is a very interesting route. You have to visit it!!!""",
    """A place you have to visit when passing through Santiago. The panoramic view is beautiful, and the sanctuary is a gorgeous spot. I recommend going up by cable car and coming down by funicular because you enjoy the view much more on the cable car ascent!""",
    """The marvelous views and the impressive waterfall here will blow you away. You have to go down many wooden stairs surrounded by stunning nature; it’s a must-visit for nature lovers. Be physically prepared for a steep climb back up, and I recommend bringing water to stay refreshed!""",
    """You really have to want to get there: You need to leave very early and bring plenty of protection against the cold... but the place is impressive. You have to go... but dress very warmly.""",
    """It's a small place, but you learn quite a lot about meteorites. There are original pieces as well as comparisons with other rocks so you know how to tell them apart. The audio guide is interesting, but I wish the museum were larger and had more pieces. Regardless, I recommend a visit if you're interested in the subject."""
]

problem_list_2 = [
    """Pros: small group van (4 couples) Beautiful volcano (Orsono) “Cheaper than Cruise ship” Cons: Weak Guide Not “All That” interesting Requested vegetarian empanada and only offered Beef to us vegetarians— ugh “NO Lunch for us”... starving by the end of 6 hour tour... with really no apology, even after pre confirming vegetarian meal in advance... Price just too high for what we received. RNelson (Austin, TX) Hello! Thanks for taking the time to write us! We are always looking to improve. The tour you took was $79 USD p/p for our 7-hour tour to Petrohue, Puerto Varas and Frutillar excursion with a maximum of 8 people in a van. Everyone that requests vegetarian empanadas beforehand gets them, I have doubled checked your e-mail and I am happy for you to resend any email where you requested a vegetarian empanada beforehand as I cannot see it. However, you did have the chance to buy anything you wanted at any stop, since there is food at every stop and minimum of 50 minutes to explore at each stop. Please elaborate as to what you mean by "weak guide" so that we may improve our services. Thanks again! Victoria Stein GV Tours""",
    """Muy bonita la vista desde la playa hacia el glaciar grey, pero sin duda alguna es mejor hacerlo en barco para poder aproximarse mucho más. Aún así la caminata por el puente colgante y el pedacito de bosque y playa es un buen panorama y más barato.""",
    """We booked our tours with ED, we closed 06/07/08 August 2024 , were super honest, wonderful tours, punctual , we returned to the hotel tired of so much to walk, can close tour of vcs with tranquility , galera is good people , driver and guide super friendly with the brazilian galera ! Note: one of the tours we won a picnic , very top congratulations Alfatur""",
    """There is a bit of confusion in buying tickets - you need to be prepared to wait if there aren't enough people to make the trip. Also, you need to understand Spanish as we did not find any offer of English translation. The boat trip had lovely scenery, and the big ships in port, and the sea lions were just as interesting to the kids as the buoy. Was also a nice idea to have a (cheap) photo waiting for us when we returned to the pier.""",
    """Tour bien organizado con audioguia. Atestado de gente, impide explorar con mas libertad en tiempo los espacios. Objetos hermosísimos no son mencionados y no hay guías en cada estar para preguntar por ellos. Cálido lugar, cargado de la esencia nerudiana. Una casa convertida en navío y locomotora...Deja sin palabras.""",
    """Grandes áreas verdes para pasear y descansar. Mucha gente haciendo deporte, haciendo picnic, o incluso durmiendo una siesta bajo los árboles.""",
    """Este es un verdadero museo...aunque bien abandonado, como muchos lugares históricos de nuestro País. Ello sin embargo, no es obstáculo para que muchos turistas lo visiten a diario, aprovechando las benditas bondades de nuestro Norte Grande, en el cual el clima invita al paseante diariamente. Lo visité en 2005 y 2012.""",
    """The place has spectacular views. We went up past the bridge. The climb is not easy at all, considering that we are seniors, but calmly, it is possible. I arrived at a very nice pool where the force of the waterfall can be seen.""",
    """Es un Mall más, salvo los perfumes (algunos) el licor en un 30% más barato y algunos lentes, todo vale lo mismo o incluso más caro que en Santiago, ojo con las falsificaciones que se venden como originales.""",
    """Creo que a Pablo, que gustaba de la compañia de sus amigos, no le gustaría mucho esta situación impersonal que resulta del audio, sería maravilloso que volvieran los guías humanos en persona. El resto siempre es interesante""",
    """Piece of history very important to the Magallanes Region. Local initiative that promotes local history""",
    """Excelente experienca como para ir denuevo, se agradece lo puntual y la amabilidad de los guias, los vehiculos son muy comodos, Quiero agregar que Felipe Carvajal tenia amplio conocimiento sobre la historia del lugar, lo cual demuestra profesionalismo en su trabajo, muy buena atencion desde que entramos a la agencia hasta que hicimos el tour saludos.""",
    """Hola para quienes se interesan por conocer este lindo atractivo turístico cercano a Pucón y que aun es desconocido para muchos quisiera comentarles que la entrada esta frente al aeródromo y debes entrar a pie o en vehículo por el letrero que dice cabañas nativas hacia arriba y llegaras a un portón de madera desde ahí es mas sencillo acceder ya que por el otro camino no esta señalizado este acceso es mas sencillo y tiene guiás ahora si vas con niños les recomiendo que todos lleven cortavientos y zapatillas de agua así es mas fácil atravesar los ríos que no son caudalosos de otra manera tendrías que irte saltando de piedra en piedra y aveces con niños o adultos mayores esa idea es nula les comento esto por que fui con mis hijos, el salto es maravilloso las fotografiás no le hacen justicia a lo que realmente es en vivo les invito hacer en familia el recorrido es una experiencia inolvidable SI LES INTERESA LES DEJO EL NOMBRE DEL ENCARGADO DEL PARQUE SU NOMBRE ES LUCIANO 9-53540492 el es muy preocupado de la seguridad ademas hay un lugar de descanso donde puedes distenderte luego del recorrido lo que hace de esta experiencia algo que es difícil de olvidar. Hola Isodoraaaa! Muchísimas gracias por tu comentario, excelente explicación, que ayudó a muchas personas a llegar durante la temporada! No me olvido más de tu famila, unos divinos todos, ojalá la vida los traiga otra vez por estos lados y te prometo que cada vez la van a pasar mejor y se van a ir mas lleno de lindas anécdotas, que es lo que nosotros queremos para cada uno de nuestros visitantes! Les mando un abrazo gigante! Y gracias por confiar en nuestro proyecto, ojalá que muchas personas más tengan la oportunidad de volver y seguír llevandose esa semillita del cuidado medioambiental que tanto queremos repartir por el mundo entero! Otra vez gracias""",
    """Llegamos y no esperamos nada para comenzar nuestro tour de cata de vinos, tanto Chilenos como extranjeros disfrutamos de una hora que se hace muy breve ya que el paisaje, la explicación de la elaboración de los vinos y el viñedo mismo son dignos de disfrutar al máximo. Cerramos con una cata de vinos y una copa de regalo, lo encntre especial para vivir la experiencia enpareja""",
    """toda esa zona es increíble, lo que la naturaleza nos regala es algo maravilloso. He sacado miles de fotos de ése viaje y puedo decir que es un verdadero desierto, maravilloso, Las zonas están marcadas y a muchos lugares no se puede acceder así nomás. Muy recomendable! Me encantó Antofagasta Calama y San Pedro de Atacama. Caminos muy riesgosos, mucho cuidado.""",
    """Se me hizo un espacio que necesita un poco de trabajo para mejorar , mostraba deterioro , y es más la FAMA que otra cosa pues con un buen tour que ofrecieran y que mostrarán fotos o placas de los presentados donde se viera su gran historia uno quedaría contento, nuestro guía nos dio un gran tour pero falto""",
    """Excelentes saltos. El pozo azul es algo digo de ver. Vale la pena el viaje a conocerlo, y el costo de acceso es razonable""",
    """pareciera que la historia de chile comenzo en 1973 y no menciona lo que paso en chile antes y porque se llego al pronunciamiento militar""",
    """La maravillosa vista y e impresionante caída de agua de este salto te sorprenderá, se debe bajar muchas escalas de madera en medio de una sorprendente naturaleza, para los amantes de la naturaleza una visita imperdible. Ir preparado físicamente para realizar un retorno de mucha subida y se recomienda llevar agua para refrescarte""",
    """Es un lugar pequeño pero aprendes bastante de meteoritos. Hay piezas originales y comparativas con otras piedras para no confundirlos. Interesante la audio guía, pero me hubiera gustado que fuera más grande y con más piezas. De todas maneras recomiendo visitar si te interesa el tema."""
]

for problem in problem_list:
    result = analyze_data(
        problem,
        model,
        tokenizer
    )
    
    print(f"Review: {problem}")
    print(f"Extracted JSON: {result}")
    print("-" * 80)

Review: There is a bit of confusion in buying tickets - you need to be prepared to wait if there aren't enough people to make the trip. Also, you need to understand Spanish as we did not find any offer of English translation. The boat trip had lovely scenery, and the big ships in port, and the sea lions were just as interesting to the kids as the buoy. Was also a nice idea to have a (cheap) photo waiting for us when we returned to the pier.
Extracted JSON: {'razonamiento_corto': 'Step 1: management (direct address to service improvements). Step 2: future gap (lack of English translation, unclear ticket process). Step 3: actionable (request for English support and clearer ticket instructions). SUGGESTION', 'etiqueta': 'SUGGESTION'}
--------------------------------------------------------------------------------
Review: An amazing place... As soon as you arrive in San Pedro, you start to notice the mountain range where the Moon Valley is located, leaving you with the feeling that you hav